# Understand embeddings with Word2Vec

### Exercise objectives:
- Convert 🔠 words to 🔢 vector representations thanks to embeddings
- Discover the powerful Word2Vec algorithm

<hr>

_Embeddings_ are representations of words using vectors. These embeddings can be learned within a Neural Network. But it can take time to converge. Another option is to learn them as a first step. Then, use them directly to feed the word representations into a Recurrent Neural Network. 

▶️ Run this cell and make sure the version of 📚 [Gensim - Word2Vec](https://radimrehurek.com/gensim/auto_examples/index.html) you are using is ≥ 4.0!

In [1]:
!pip freeze | grep gensim

gensim==4.3.3


In [2]:
!pip freeze | grep tensorflow

tensorflow==2.16.2
tensorflow-datasets==4.9.7
tensorflow-metadata==1.16.1


# The data

Keras provides many datasets, among which is the IMDB dataset 🎬:
- It is comprised of sentences that are ***movie reviews***. 
- Each of these reviews is related to a score given by the reviewer.

❓ **Question** ❓ First of all, let's load the data. You don't have to understand what is going on in the function, it does not matter here.

⚠️ **Warning** ⚠️ The `load_data` function has a `percentage_of_sentences` argument. Depending on your computer, there are chances that too many sentences will make your compute slow down, or even freeze - your RAM can overflow. For that reason, **you should start with 10% of the sentences** and see if your computer can handle it. Otherwise, rerun with a lower number.  

⚠️ **DISCLAIMER** ⚠️ **No need to play _who has the biggest_ (RAM) !** The idea is to get to run your models quickly to prototype. Even in real life, it is recommended that you start with a subset of your data to loop and debug quickly. So increase the number only if you are into getting the best accuracy. 

In [3]:
###########################################
### Just run this cell to load the data ###
###########################################

import tensorflow_datasets as tfds
from tensorflow.keras.preprocessing.text import text_to_word_sequence

def load_data(percentage_of_sentences=None):
    train_data, test_data = tfds.load(name="imdb_reviews", split=["train", "test"], batch_size=-1, as_supervised=True)

    train_sentences, y_train = tfds.as_numpy(train_data)
    test_sentences, y_test = tfds.as_numpy(test_data)

    # Take only a given percentage of the entire data
    if percentage_of_sentences is not None:
        assert(percentage_of_sentences> 0 and percentage_of_sentences<=100)

        len_train = int(percentage_of_sentences/100*len(train_sentences))
        train_sentences, y_train = train_sentences[:len_train], y_train[:len_train]

        len_test = int(percentage_of_sentences/100*len(test_sentences))
        test_sentences, y_test = test_sentences[:len_test], y_test[:len_test]

    X_train = [text_to_word_sequence(_.decode("utf-8")) for _ in train_sentences]
    X_test = [text_to_word_sequence(_.decode("utf-8")) for _ in test_sentences]

    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = load_data(percentage_of_sentences=10)

2026-05-19 14:27:41.247298: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-19 14:27:41.249538: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-19 14:27:41.260161: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-19 14:27:41.279854: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-19 14:27:41.305657: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

<b><u>Embeddings in the previous challenge</u></b>:

In the previous exercise, we jointly learned a representation for the words, and fed this representation to a RNN, as shown down below 👇: 

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/06-DL/NLP/layers_embedding.png" alt="Joint learning of embedding and RNN" width="400px" />

However, this increases the number of parameters to learn, which slows down and increases the difficulty of convergence!

<b><u>Embeddings in the current challenge</u></b>:

For this reason, we will separate the steps of learning the word representation and feeding it into a RNN. As shown here: 

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/06-DL/NLP/word2vec_representation.png" alt="Training of RNN and embedding independently" width="400px" />

We will learn the embedding with Word2Vec.

The drawback is indeed that the learned embeddings are not _specifically_ designed for our task. However, learning them independently of the task at hand (sentiment analysis) has some advantages: 
- it is very fast to do in general (with Word2Vec)
- the representation learned by Word2Vec is still meaningful 
- the convergence of the RNN alone will be easier and faster

So let's learn an embedding with Word2Vec and see how meaningful it is!

# Embedding with Word2Vec

Let's use Word2Vec to embed the words of our sentences. Word2Vec will be able to convert each word to a fixed-size vectorial representation.

For instance, we will have:
- 🐶 _dog_ $\rightarrow$ [0.1, -0.3, 0.8]
- 🐱 _cat_ $\rightarrow$ [-1.1, 2.3, 0.7]
- 🍏 _apple_ $\rightarrow$ [3.1, 0.9, -4.7]

Here, your embedding space is of size 3.

***What is a "good" numerical representation of words?***

- ***Words with close meanings should be geometrically close in your embedding space!***

    - Look at the following example which represents a bi-dimensional embedding space.

![Embedding](https://wagon-public-datasets.s3.amazonaws.com/data-science-images/06-DL/NLP/word_embedding.png)

❓ **Question** ❓ Let's run Word2Vec! 

[📚 **Gensim**](https://radimrehurek.com/gensim/)  is a great Python package that makes the use of the Word2Vec algorithm easy to implement, fast and accurate (which is not an easy task!).

1. The following code imports Word2Vec from Gensim. 

2. The second line learns the embedding representation of the words thanks to the sentences in `X_train`. 
3. The third line stores the words and their trained embeddings in `wv`.

In [4]:
from gensim.models import Word2Vec

word2vec = Word2Vec(sentences=X_train)
wv = word2vec.wv

Let's look at the embedded representation of some words.

You can use `wv` as a dictionary.
For instance, `wv['dog']` will return a representation of `dog` in the embedding space.

❓ **Question** ❓ Try different words - especially, try non-existing words to see that they don't have any representation (which is perfectly normal as their representation was not learned). 

In [6]:
# YOUR CODE HERE
print(wv['movie'])
print(wv['good'])

[ 0.6123102   0.0687464  -0.12449975 -0.08015873  1.5106115  -1.612622
  1.245273    1.4509944  -1.2495849  -1.0621159  -0.73705125 -0.46427003
  1.0226454   1.1410228   0.945997   -1.266246    0.5968486   0.7262043
  0.35951906 -2.233336    1.1470824  -0.68804455  0.33390072 -0.5835164
  0.36117098 -0.68715644 -2.520827   -0.5222678   0.28357577 -0.0232139
  1.5224012   0.8915697  -0.99437684  0.08448157 -0.23432688  1.1870407
 -2.2946525   1.5469196   0.93670315 -1.5965556   0.6872418  -0.27765352
  1.9027032   0.81732696 -0.43339652  0.22742556 -0.60098314 -0.68552154
  0.24211137 -0.3228977  -0.08927346  0.6742987  -0.27133736  0.05123157
  1.1190974  -0.45186     0.6443852  -0.0953603  -0.7914959  -0.07464409
 -0.48280865 -0.34051505  0.8099341   0.8157749  -1.2384036   0.8945182
  1.0440999  -0.01634723 -0.5284938  -0.56540614  0.77439225 -0.29710814
  0.57751024  0.68825865  1.3980433  -0.354231    0.25832698  1.0045483
 -0.2672508   1.0725964  -0.46326515  0.74432635 -0.0999361

❓ **Question** ❓ What is the size of each word representation, and therefore, what is the size of the embedding space?

In [7]:
# YOUR CODE HERE
wv['movie'].shape

(100,)

🧐 How do we know whether this embedding make any sense or not? 

💡 To investigate this question, we will check that words with a close meaning have close representations. 

👉 Let's use the [**`Word2Vec.wv.most_similar`**](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.most_similar) method that, given an input word, displays the "closest" words in the embedding space. If the embedding is well done, then words with similar meanings will have similar representation in the embedding space.

❓ **Question** ❓ Try out the `most_similar` method on different words. 

🧑🏿‍🏫 The quality of the closeness will depend on the quality of your embedding, and thus, depend on the number of sentences that you have loaded and from which you create your embedding.

In [17]:
# YOUR CODE HERE
wv.most_similar('good')

[('great', 0.923400342464447),
 ('funny', 0.9050050377845764),
 ('bad', 0.876315712928772),
 ('quite', 0.8660200238227844),
 ('very', 0.8289985656738281),
 ('awful', 0.8177327513694763),
 ('nice', 0.8145514726638794),
 ('terrible', 0.8077070116996765),
 ('pretty', 0.7959688305854797),
 ('simply', 0.7951465249061584)]

📚 Similarly to `most_similar` used on words directly, we can use [**`similar_by_vector`**](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.similar_by_vector) on vectors to do the same thing:

In [16]:
# YOUR CODE HERE

wv.similar_by_vector(wv['good'])

[('good', 1.0),
 ('great', 0.923400342464447),
 ('funny', 0.9050050377845764),
 ('bad', 0.876315712928772),
 ('quite', 0.8660200238227844),
 ('very', 0.8289985656738281),
 ('awful', 0.8177327513694763),
 ('nice', 0.8145514726638794),
 ('terrible', 0.8077070713043213),
 ('pretty', 0.7959688305854797)]

# Arithmetic on words

Now, let's perform some mathematical operations on words, i.e. on their vector representations!

As any learned word is represented as a vector, you can do basic arithmetic operations, such as:

$$W2V(good) - W2V(bad)$$

❓ **Question** ❓ Do this mathematical operation and print the result

In [18]:
# YOUR CODE HERE
result = wv['good'] - wv['bad']

print(result)

[-0.12487739 -0.23642553  0.04700503  0.28934407  0.00978333 -0.6303011
 -0.08567821 -0.19215104 -0.30190396 -0.02125239  0.15571235 -0.30153006
  0.4370441   0.05206919 -0.03164873  0.23696388 -0.41251498  0.5460352
 -0.35649592 -0.25559044 -0.00506151 -0.04106537  0.14624286 -0.2391577
 -0.27091372 -0.05186754 -0.23011273  0.8078082  -0.25229585 -0.68964976
  0.05792058  0.308309    0.03692997  0.4739685  -0.12031817 -0.21635592
  0.3953806  -0.03636098 -0.5083711   1.1337136   0.04345805 -0.0284303
 -0.601617   -0.27303946  0.41569328  0.03253239  0.33542696 -0.13522339
  0.07343155  0.6201495   0.09310347 -0.1790914  -0.16079943  0.00338846
  0.22900727  0.097977   -0.16410118  0.16259383  0.14223519  0.3593387
 -0.01511733  0.3526106  -0.24634844 -0.2534821   0.05420625 -0.1442442
 -0.6852812  -0.18182948  0.5733905   0.5410287  -0.7006932  -0.4778538
  0.3678161  -0.04381549 -0.22870249 -0.02020732  0.11972081 -0.12108421
  0.24935767  0.10818854  0.27399296 -0.23611994 -0.488145

Now, imagine for a second that the following equality holds true:

$$W2V(good) - W2V(bad) = W2V(nice) - W2V(stupid)$$

which is equivalent to:

$$W2V(good) - W2V(bad) + W2V(stupid) = W2V(nice)$$

❓ **Question** ❓ Let's, just for fun (as it would be bold of us to think that this equality holds true ...), do the operation $W2V(good) - W2V(bad) + W2V(stupid)$ and store it in a `res` variable (which will be a vector of size 100 that you can print).

In [19]:
# YOUR CODE HERE
res = wv['good'] - wv['bad'] + wv['stupid']

print(res)

[-7.27059692e-03 -2.59788841e-01  1.46371216e-01 -1.05929881e-01
 -3.56124714e-02 -1.39360964e+00  9.20830294e-02  2.62481093e-01
 -8.27456534e-01 -3.15986902e-01 -1.27617106e-01 -6.47889495e-01
  4.29200709e-01  3.53337348e-01  1.96385264e-01  3.95790040e-02
 -2.38205969e-01  1.70713395e-01 -3.95718873e-01 -1.33314955e+00
  4.66894895e-01  4.73888814e-02  8.47304285e-01 -2.80110955e-01
 -5.38658023e-01 -5.93191646e-02 -7.42317557e-01  6.48184538e-01
 -2.70986706e-01 -5.72631598e-01  6.68816626e-01  5.51932693e-01
  4.15968657e-01  2.19390392e-01 -4.43540126e-01  3.36569607e-01
  4.98251319e-02  1.75046116e-01 -6.37666464e-01  3.90341163e-01
  1.30691290e-01 -3.95854354e-01 -3.10769796e-01  2.55935073e-01
  7.65739858e-01 -5.02850860e-04  5.02667725e-02 -3.96887392e-01
 -1.89771205e-01  6.53951705e-01  3.67872685e-01 -5.36180019e-01
 -3.30450416e-01  8.13370124e-02  2.36535102e-01  2.81779051e-01
  1.48896247e-01  2.69565523e-01 -3.58586162e-01  2.35239252e-01
  1.11339644e-01  2.45315

We said earlier, that for any vector it is possible to see the closest vectors in the embedding space.

❓ **Question** ❓ Look at the closest vectors of `res`

💡 _Hint_: `similar_by_vector`

In [20]:
# YOUR CODE HERE
wv.similar_by_vector(res)

[('good', 0.7883428931236267),
 ('nice', 0.7688433527946472),
 ('always', 0.754597008228302),
 ('great', 0.7456106543540955),
 ('also', 0.7392008900642395),
 ('done', 0.7318249344825745),
 ('given', 0.7190194129943848),
 ('potential', 0.7108736038208008),
 ('although', 0.7052227258682251),
 ('considered', 0.7052013874053955)]

Incredible right! You can do arithmetic operations on words!

❓ **Question** ❓ You can try on arithmetic such as 

$$W2V(Boy) - W2V(Girl) = W2V(Man) - W2V(Woman)$$

or 

$$W2V(Queen) - W2V(King) = W2V(actress) - W2V(actor)$$

❗ **Remark** ❗ You will probably see that the results are not perfect. But don't forget that you trained your model on a very small corpus.

In [21]:
# YOUR CODE HERE
wv['boy'] - wv['girl'] + wv['woman']

array([-5.8797514e-01,  3.4836271e-01, -1.6422020e-01,  5.2353245e-01,
       -1.7651939e-01, -3.5233909e-01,  2.5270537e-01,  1.1244550e+00,
       -1.6720679e-01, -2.7697957e-01,  1.9049102e-01, -6.4978182e-01,
       -2.4701145e-01, -9.3518570e-02, -2.3843507e-01, -1.8040136e-02,
        3.5273880e-03, -9.2708416e-02,  2.7096465e-02, -5.0337493e-01,
       -1.6941446e-01,  1.5402858e-01,  2.9625857e-01, -5.3070325e-01,
        7.7279143e-02, -1.5540975e-01, -5.1081407e-01, -6.5194964e-03,
       -4.8729753e-01, -1.9252813e-01,  3.5916838e-01, -1.7636856e-01,
        2.6176944e-01, -8.1588608e-01, -4.0769911e-01,  1.8825199e-01,
        4.6465144e-01, -4.9067146e-01, -5.9227604e-01,  2.5394008e-02,
       -1.2049228e-03, -4.9345633e-01, -1.0500391e+00, -9.4648525e-02,
        6.6530031e-01, -1.6668031e-01, -1.3098964e-01,  3.7412103e-02,
        9.8894441e-01,  3.8683030e-01,  2.9445833e-01, -4.5349166e-01,
        1.2272775e-02, -5.1467817e-02, -2.1801689e-01,  2.4143091e-01,
      

<u><i>Some notes about Word2Vec as an internal Neural Network</i></u>:

You might wonder where does this magic comes from (at quite a low price, you just ran a line of code on a very small corpus and it was trained within few minutes). The magic comes from the way Word2Vec is trained. The details are quite complex, but you can remember that Word2vec, in `word2vec = Word2Vec(sentences=X_train)`, actually trains a internal neural network (that you don't see).  

In a nutshell, this internal neural network predicts a word from the surroundings words in a sentences. Hence, it splits the original sentences, then for each split it chooses some words as inputs $X$ and a word as the output $y$ which it tries to predict, using the embedding space.

And as with any neural network, Word2Vec has some hyperparameters. Let's play with some of these. 

# Word2Vec hyperparameters

❓ **Question** ❓ The first important hyperparameter is the `vector_size` argument. It corresponds to the size of the embedding space. Learn a new `word2vec_2` model, still trained on the `X_train`, but with a smaller or higher `vector_size`.

Verify on some words that the embedding size is the one you chose.

In [22]:
# YOUR CODE HERE
word2vec_2 = Word2Vec(sentences=X_train, vector_size=50)

wv_2 = word2vec_2.wv

wv_2['movie'].shape

(50,)

❓ **Question** ❓ Use the **`Word2Vec.wv.key_to_index`** attribute to display the size of the learned vocabulary. Compare it to the number of different words in `X_train`.

In [23]:
# YOUR CODE HERE
word2vec_vocab_size = len(wv.key_to_index)

train_vocab = set([word for sentence in X_train for word in sentence])
train_vocab_size = len(train_vocab)

print(word2vec_vocab_size)
print(train_vocab_size)

8006
30419


There is an important difference between the number of words in the train sentences and in the Word2Vec vocabulary, even though it has been trained on the train sentence set. The reasons comes from the second important hyperparameter of Word2Vec:  `min_count`. 

`min_count` is a integer that tells you how many occurrences a given word should have to be learned in the embedding space. For instance, let's say that the word "movie" appears 1000 times in the corpus and "simba" only 2 times. If `min_count=3`, the word "simba" will be skipped during the training.

The intention is to learn a representation of words that are sufficiently present in the corpus to have a robust embedded representation.

❓ **Question** ❓ Learn a new `word2vec_3` model with a `min_count` higher than 5 (which is the default value) and a `word2vec_4` with a `min_count` smaller than 5, and then, compare the size of the vocabulary for all the different word2vecs that you have trained (you can choose any `vector_size` you want).

In [24]:
# YOUR CODE HERE
word2vec_3 = Word2Vec(sentences=X_train, min_count=10)
word2vec_4 = Word2Vec(sentences=X_train, min_count=1)

print(len(word2vec.wv.key_to_index))
print(len(word2vec_3.wv.key_to_index))
print(len(word2vec_4.wv.key_to_index))

8006
4503
30419


Remember that Word2Vec has an internal neural network that is optimized based on some predictions. These predictions actually correspond to predicting a word based on surrounding words. The surroundings words are in a `window` which corresponds to the number of words taken into account. And you can train the Word2Vec with different `window` sizes.

❓ **Question** ❓ Train a new `word2vec_5` model with a `window` different than previously (default is 5).

In [25]:
# YOUR CODE HERE
word2vec_5 = Word2Vec(sentences=X_train, window=10)

The arguments you have seen (`vector_size`, `min_count` and `window`) are usually the ones that you should start playing with to get a better performance for your model.

But you can also look at other arguments in the [**📚 Documentation - gensim.models.word2vec.Text8Corpus**](https://radimrehurek.com/gensim/models/word2vec.html#gensim.models.word2vec.Text8Corpus)

# Convert our train and test set to RNN-ready datasets

Remember that `Word2Vec` is the first step to the overall process of feeding such a representation into a RNN, as shown here:

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/06-DL/NLP/word2vec_representation.png" alt="Word2Vec and RNN" width="400px" />



Now, let's work on Step 2 by converting the training and test data into their vector representation to be ready to be fed in RNNs.

❓ **Question** ❓ Now, write a function that, given a sentence, returns a matrix that corresponds to the embedding of the full sentence, which means that you have to embed each word one after the other and concatenate the result to output a 2D matrix (make sure that your output is a NumPy array)

❗ **Remark** ❗ You will probably notice that some words you are trying to convert throw errors as they are said not to belong to the dictionary:

- For the <font color=orange>test</font> set, this is understandable: <font color=orange>some words were not</font> in the <font color=blue>train</font> set and thus, their <font color=orange>embedded representation is unknown</font>
- for the <font color=blue>train set</font>, due to `min_count` hyperparameter, not all the words have a vector representation.

In any case, just skip the missing words here.

In [ ]:
import numpy as np

example = ['this', 'movie', 'is', 'the', 'worst', 'action', 'movie', 'ever']
example_missing_words = ['this', 'movie', 'is', 'laaaaaaaaaame']

def embed_sentence(word2vec, sentence):
    # YOUR CODE HERE
    embedded_sentence = []

    for word in sentence:
        if word in word2vec.wv:
            embedded_sentence.append(word2vec.wv[word])
    return np.array(embedded_sentence)


### Checks
embedded_sentence = embed_sentence(word2vec, example)
assert(type(embedded_sentence) == np.ndarray)
assert(embedded_sentence.shape == (8, 100))

embedded_sentence_missing_words = embed_sentence(word2vec, example_missing_words)
assert(type(embedded_sentence_missing_words) == np.ndarray)
assert(embedded_sentence_missing_words.shape == (3, 100))

❓ **Question** ❓ Write a function that, given a list of sentences (each sentence being a list of words/strings), returns a list of embedded sentences (each sentence is a matrix). Apply this function to the train and test sentences

💡 _Hint_: Use the previous function `embed_sentence`

In [30]:
def embedding(word2vec, sentences):
    # YOUR CODE HERE
    embedded_sentences = []

    for sentence in sentences:
        embedded_sentences.append(embed_sentence(word2vec, sentence))

    return embedded_sentences

X_train_embedded = embedding(word2vec, X_train)
X_test_embedded = embedding(word2vec, X_test)

X_train_embedded = embedding(word2vec, X_train)
X_test_embedded = embedding(word2vec, X_test)

❓ **Question** ❓ In order to have ready-to-use data, do not forget to pad your sequences so you have tensors which can be divided into batches (of `batch_size`) during the optimization. Store the padded values in `X_train_pad` and `X_test_pad`. Do not forget the important arguments of the padding ;)

In [31]:
### YOUR CODE HERE
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_pad = pad_sequences(
    X_train_embedded,
    dtype='float32',
    padding='post',
    truncating='post'
)

X_test_pad = pad_sequences(
    X_test_embedded,
    dtype='float32',
    padding='post',
    truncating='post'
)
assert(len(X_train_pad.shape) == 3)
assert(len(X_test_pad.shape) == 3)
assert(X_train_pad.shape[2] == 100)
assert(X_test_pad.shape[2] == 100)



🏁 Congratulations, you are now able to use `Word2Vec` to embed your words :)

💾 Don't forget to git add/commit/push your notebook...

🚀 ... and move on to the next challenge!
